# Milestone 2: RAG Exploration

This notebook has the initial explorations of the RAG pipeline.

In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from langchain_groq import ChatGroq
import faiss
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from groq import Groq
import os
import sys
import os

sys.path.append(os.path.abspath(".."))

/Users/yasamanbaher/miniforge3/envs/project-575/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# -------------------------
# Load data
# -------------------------
meta = pd.read_json("../data/raw/meta_Toys_and_Games.jsonl", lines=True, nrows=50000)
review = pd.read_json("../data/raw/Toys_and_Games.jsonl", lines=True, nrows=50000)

cleaned_meta = meta.drop(columns=['videos', 'price', 'images', 'bought_together', 'subtitle', 'author'], errors='ignore')
cleaned_meta.head()

reviews = review[review['verified_purchase'] == True]
cleaned_reviews = reviews.drop(columns=['images', 'timestamp', 'user_id', 'verified_purchase'], errors='ignore')
cleaned_reviews.head()

,rating,title,text,asin,parent_asin,helpful_vote
0,5,Granddaughters love them!,I purchased several of these for my granddaugh...,B09QH7QJS7,B09QH7QJS7,0
1,3,Arrived broken,"It’s cute, but it arrived broken & has some pr...",B06XYKSKQP,B06XYKSKQP,1
2,5,Dylan loves them!!!,Bough for my grandson Dylan. He loves them.,B07SFF3YQW,B07XRSD5R9,0
3,5,Janod quality and very cute...,This was great for my daughters circus themed ...,B007JWWUDW,B007JWWUDW,0
4,3,I used for my daughters circus birthday party ...,This is very cute but beware that the animals ...,B00MZG6OO8,B00MZG6OO8,2


In [3]:
# -------------------------
# Clean text columns
# -------------------------
cleaned_meta['description'] = cleaned_meta['description'].apply(
    lambda x: " ".join(x) if isinstance(x, list) else (x if isinstance(x, str) else "")
).str.lower()

cleaned_meta['details'] = cleaned_meta['details'].apply(
    lambda x: " ".join([f"{k} {v}" for k, v in x.items()]) if isinstance(x, dict) else ""
).str.lower()

cleaned_meta['features'] = cleaned_meta['features'].apply(
    lambda x: " ".join(x) if isinstance(x, list) else ""
).str.lower()

cleaned_meta['categories'] = cleaned_meta['categories'].apply(
    lambda x: " ".join(x) if isinstance(x, list) else ""
).str.lower()

cleaned_meta['title'] = cleaned_meta['title'].str.lower()

cleaned_meta = cleaned_meta[
    (cleaned_meta['title'].str.strip() != '') &
    (cleaned_meta['description'].str.strip() != '') &
    (cleaned_meta['features'].str.strip() != '') &
    (cleaned_meta['categories'].str.strip() != '')
].reset_index(drop=True)

In [4]:
# -------------------------
# Prepare review text per product
# -------------------------
cleaned_reviews = cleaned_reviews.copy()
review_text_cols = [col for col in ['title', 'text'] if col in cleaned_reviews.columns]
cleaned_reviews['combined_review_text'] = cleaned_reviews[review_text_cols].fillna('').agg(' '.join, axis=1)
cleaned_reviews['combined_review_text'] = cleaned_reviews['combined_review_text'].str.lower()

grouped_reviews = (
    cleaned_reviews.groupby('parent_asin')['combined_review_text']
    .apply(lambda x: " ".join(x.astype(str)))
    .reset_index()
)

rag_df = cleaned_meta.merge(grouped_reviews, on='parent_asin', how='left')
rag_df['combined_review_text'] = rag_df['combined_review_text'].fillna('')

In [5]:
# -------------------------
# Build product documents
# -------------------------
products = (
    rag_df['title'] + ' ' +
    rag_df['description'] + ' ' +
    rag_df['features'] + ' ' +
    rag_df['categories'] + ' ' +
    rag_df['combined_review_text']
).tolist()

In [6]:
# -------------------------
# Embeddings and vector store
# -------------------------
model = SentenceTransformer("all-MiniLM-L6-v2")
product_embeddings = model.encode(products).astype("float32")

index = faiss.IndexFlatL2(product_embeddings.shape[1])
index.add(product_embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6047.81it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
# -------------------------
# Retriever
# -------------------------

def retrieve(query, top_k=5):
    query_embedding = model.encode([query]).astype("float32")
    distances, indices = index.search(query_embedding, top_k)
    return rag_df.iloc[indices[0]]

# -------------------------
# Building context
# -------------------------
# Implemented with the help of chatGPT
def build_context(docs):
    blocks = []
    for _, row in docs.iterrows():
        review_snippet = row.get('combined_review_text', '')[:500]

        block = (
            f"Product ASIN: {row.get('parent_asin', 'N/A')}\n"
            f"Title: {row.get('title', '')}\n"
            f"Description: {row.get('description', '')}\n"
            f"Features: {row.get('features', '')}\n"
            f"Categories: {row.get('categories', '')}\n"
            f"Review Evidence: {review_snippet}\n"
        )
        blocks.append(block)
    return "\n\n".join(blocks)


# -------------------------
# Wrapper function
# -------------------------
def retrieve_and_build_context(query):
    docs = retrieve(query, top_k=5)
    return build_context(docs)

In [8]:
# -------------------------
# Prompt variants
# -------------------------
prompt1 = ChatPromptTemplate.from_template(

"""
You must answer using ONLY the information in the context.

- If the answer is present, extract and summarize it clearly.
- Do NOT say "I don't know" if the answer exists in the context.
- Only say "I don't know" if the context truly does not contain the answer.

Context:
{context}

Question:
{input}

Answer:
"""
)

prompt2 = ChatPromptTemplate.from_template(

"""
You must answer using ONLY the information in the context.

- Keep the answer shorter than 3 sentences.
- Make sure nothing is repeated, and only necessary details are written.
- If the answer is not in the context, say: "The context does not provide enough information."

Context:
{context}

Input:
{input}

Answer:
"""
)

prompt3 = ChatPromptTemplate.from_template(

"""
You must answer using ONLY the information in the context.

- Be clear and helpful
- Give specific statements instead of general ones. 
- If something is missing, say "not enough context to answer your question"

Context:
{context}

Question:
{input}

Answer:
"""
)

In [ ]:
# please uncomment this line below and add your GROQ API key before proceeding
# import os
# os.environ["GROQ_API_KEY"] = "YOUR KEY HERE"

In [19]:
# -------------------------
# Rag Pipeline model 1
# -------------------------
# Implemented with the help of chatGPT

llm1 = ChatGroq(model="llama-3.1-8b-instant")

prompts = {
    "prompt1": prompt1,
    "prompt2": prompt2,
    "prompt3": prompt3
}

query = "A good board game for kids age 8 and up"

for name, prompt in prompts.items():
    test_chain = (
        {
            "context": RunnableLambda(retrieve_and_build_context),
            "input": RunnablePassthrough()
        }
        | prompt
        | llm1
        | StrOutputParser()
    )

    print(f"\n===== {name} =====")
    print(test_chain.invoke(query))

rag_chain = (
    {
        "context": RunnableLambda(retrieve_and_build_context),
        "input": RunnablePassthrough()
    }
    | prompt1
    | llm1
    | StrOutputParser()
)

queries = [
    "A good board game for kids age 8 and up",
    "A toy for toddlers",
    "Educational toys for kids",
    "Toys for learning colors",
    "A toy for sensory development"
]

for q in queries:
    print(f"\nQUERY: {q}")
    print(rag_chain.invoke(q))


===== prompt1 =====
Based on the context, a good board game for kids age 8 and up could be the "Sorry!" game. 

This is mentioned in the description: 
"This kind of frustration may be hard for children under age 8 to handle. in fact, young ones typically crumble into tears of outrage when their pawns are cavalierly sent back. the only recourse is to teach children how to plot their own revenge, which makes them feel as powerful as superheroes."

However, another option could be the "Chess, Checkers & Tic Tac Toe" game, which is recommended for ages 6 and up, but it's still a suitable option for kids 8 and up.

It's also mentioned in the description of the "Sorry!" game: 
"this classic game of luck, strategy, and determination is easy to grasp for children as young as 6 years old, yet it's fun for adults and older siblings too."

===== prompt2 =====
The Sorry! board game is suitable for kids as young as 6 years old, but children under age 8 may have trouble handling the frustration of 

In [ ]:
# -----------------------------------------
# Rag Pipeline model 2 for final submission
# -----------------------------------------

llm2 = ChatGroq(model="llama-3.3-70b-versatile")

prompts = {
    "prompt1": prompt1,
    "prompt2": prompt2,
    "prompt3": prompt3
}

query = "A good board game for kids age 8 and up"

for name, prompt in prompts.items():
    test_chain = (
        {
            "context": RunnableLambda(retrieve_and_build_context),
            "input": RunnablePassthrough()
        }
        | prompt
        | llm2
        | StrOutputParser()
    )

    print(f"\n===== {name} =====")
    print(test_chain.invoke(query))

rag_chain = (
    {
        "context": RunnableLambda(retrieve_and_build_context),
        "input": RunnablePassthrough()
    }
    | prompt1
    | llm2
    | StrOutputParser()
)

queries = [
    "A good board game for kids age 8 and up",
    "A toy for toddlers",
    "Educational toys for kids",
    "Toys for learning colors",
    "A toy for sensory development"
]

for q in queries:
    print(f"\nQUERY: {q}")
    print(rag_chain.invoke(q))


===== prompt1 =====
The Sorry! board game is suitable for kids ages 6 and up, but it's mentioned that children under age 8 may find it frustrating when their pawns are sent back to the starting line. However, it can still be a good option for kids age 8 and up. 

Another option is the Chess, Checkers & Tic Tac Toe game, which is recommended for ages 6 and up.

===== prompt2 =====
The Sorry! board game is suitable for kids ages 6 and up, making it a good option for kids age 8 and up.

===== prompt3 =====
The "Sorry! Board Game" (ASIN: B00000IWD0) is a good option for kids age 8 and up. It's a classic game of strategy, chance, and luck that is easy to grasp for children as young as 6 years old, yet fun for adults and older siblings too. 

Another option is the "Chess, Checkers & Tic Tac Toe" game (ASIN: B07P2VVFVM), which is recommended for ages 6 and up and includes three classic games in one convenient package.

QUERY: A good board game for kids age 8 and up
The "Sorry!" board game is